# 01 — Exploratory Data Analysis

**This notebook is not the source of truth.** Every number that matters is
produced by `src/` and reproduced by `make baseline`. This is for looking at the
data, not for deciding anything.

Covers: population shape, target distribution under the performance window,
missingness, vintage stability, and the univariate signal available to Phase 1.


In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from src.config import SYNTHETIC_SPLIT, SYNTHETIC_TARGET
from src.ingestion.loaders import describe_sources, load
from src.ingestion.splits import split_by_time, vintage_column
from src.ingestion.target import assign_labels_from_dpd, label_summary, modelling_population

pl.Config.set_tbl_rows(25)
describe_sources()

## 1. Population and target

Labels come from the binding rule in `docs/target_definition.md`.

In [ ]:
app = load("application")
labelled = vintage_column(assign_labels_from_dpd(app, SYNTHETIC_TARGET))
label_summary(labelled)

The `censored` share is the cost of the performance window: those loans were
originated too recently for a 12-month outcome to exist. They are dropped rather
than assumed good.

In [ ]:
model_pop = modelling_population(labelled).collect()
print(f"modelling population: {model_pop.height:,} rows")
print(f"observed bad rate:    {model_pop['label'].mean():.2%}")

## 2. Bad rate by vintage

The reason splits are out-of-time. If this line were flat, a random split would be harmless — it is not flat.

In [ ]:
by_vintage = (
    model_pop.group_by("vintage")
    .agg(n=pl.len(), bad_rate=pl.col("label").mean())
    .sort("vintage")
)
by_vintage

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(by_vintage["vintage"], by_vintage["bad_rate"], marker="o")
ax.axhline(model_pop["label"].mean(), ls="--", lw=1, color="grey", label="overall")
ax.set_ylabel("bad rate"); ax.set_xlabel("origination vintage")
ax.set_title("Bad rate by vintage — the 2022 cohort underwrites worse")
ax.legend(); plt.xticks(rotation=45); plt.tight_layout()

## 3. Missingness

`EXT_SOURCE_1` is absent for a large share of applicants, and it is **not** missing at random — thin-file applicants are likelier to lack an external bureau score. That pattern is preserved as an explicit indicator feature rather than imputed away.

In [ ]:
missing = (
    model_pop.select(
        [pl.col(c).is_null().mean().alias(c) for c in model_pop.columns]
    )
    .transpose(include_header=True, header_name="column", column_names=["null_rate"])
    .filter(pl.col("null_rate") > 0)
    .sort("null_rate", descending=True)
)
missing

In [ ]:
# Is missingness informative? Compare bad rate with and without the score.
model_pop.group_by(pl.col("EXT_SOURCE_1").is_null().alias("ext1_missing")).agg(
    n=pl.len(), bad_rate=pl.col("label").mean()
).sort("ext1_missing")

## 4. Univariate signal

Bad rate across deciles of each candidate feature. A feature whose bad rate is flat across its own range carries no univariate signal — worth knowing before Phase 2's WOE/IV work formalises it.

In [ ]:
def univariate(df: pl.DataFrame, col: str, bins: int = 10) -> pl.DataFrame:
    d = df.filter(pl.col(col).is_not_null())
    return (
        d.with_columns(bucket=(pl.col(col).rank("ordinal") * bins // (d.height + 1)))
        .group_by("bucket")
        .agg(n=pl.len(), lo=pl.col(col).min(), hi=pl.col(col).max(), bad_rate=pl.col("label").mean())
        .sort("bucket")
    )

univariate(model_pop, "EXT_SOURCE_2")

In [ ]:
for col in ["EXT_SOURCE_2", "AMT_INCOME_TOTAL", "DAYS_BIRTH"]:
    u = univariate(model_pop, col)
    plt.figure(figsize=(7, 3))
    plt.plot(u["bucket"], u["bad_rate"], marker="o")
    plt.title(f"bad rate by decile of {col}"); plt.xlabel("decile"); plt.ylabel("bad rate")
    plt.tight_layout(); plt.show()

## 5. Split sanity check

Confirms the folds are contiguous in time and non-overlapping before any model is fitted.

In [ ]:
splits = split_by_time(modelling_population(labelled), SYNTHETIC_SPLIT)
for name in ("train", "calibration", "valid", "test"):
    df = getattr(splits, name).collect()
    print(f"{name:12s} n={df.height:>7,}  "
          f"{df['origination_date'].min()} -> {df['origination_date'].max()}  "
          f"bad={df['label'].mean():.2%}")

## Takeaways carried into Phase 2

1. The bad rate moves materially by vintage, so out-of-time splitting is load-bearing, not ceremony.
2. `EXT_SOURCE_*` dominates the univariate signal, and its missingness is itself predictive.
3. Application-level fields alone reach ~0.77 AUC out-of-time. The relational tables are untouched — that is where Phase 2's lift has to come from.
